# JustBuild Foundation - FinOps Invoicing Agent

Twenty-five supplier invoices, one approval policy, one purchase-order register. This notebook runs the same **three-agent** design you drew on the board, now pointed at Finance:

**Input -> Agent 1 (Ingest & Extract) -> Agent 2 (Judge & Validate) -> Agent 3 (Communicate) -> You (the human)**

- **Agent 1** turns each messy invoice into a clean row of facts.
- **Agent 2** validates every invoice against the policy and the PO register. For money, the rules live in **code**, not in the model's opinion.
- **Agent 3** drafts a summary to the finance manager. It **never pays anything** - that stays your call.

Everything the notebook needs - the invoices, the approval policy, and the **OC Brain** - is pulled from the hosted pack. Nothing is pasted inline.

> Run the cells top to bottom. Watch for the green checkmarks.

In [ ]:
#@title 1 · Setup  { display-mode: "form" }
# Installs the few libraries we need and confirms they loaded.
!pip -q install google-generativeai pypdf requests pandas >/dev/null 2>&1
import requests, io, re, json, time
from pypdf import PdfReader
import pandas as pd
print('Setup complete - libraries ready.')

In [ ]:
#@title 2 · Configuration  { display-mode: "form" }
# 'Live Gemini' calls the real model (needs a key). 'Switch to OC Brain' skips the
# network and uses the prepared results.
import getpass

MODE = "Live Gemini"  #@param ["Live Gemini", "Switch to OC Brain"]
MODEL = "gemini-2.5-flash-lite"  #@param {type:"string"}

# Inputs and the OC Brain live here (hosted, not pasted into the notebook):
BASE = "https://justbuild.orangecaterpillar.com/foundation-material/invoices"
INVOICES_URL = f"{BASE}/Invoices.pdf"
POLICY_URL   = f"{BASE}/Approval_Policy.pdf"
OC_BRAIN_URL = f"{BASE}/oc-brain.json"

# API key is only needed for Live Gemini. Typed in hidden; never saved in the notebook.
GEMINI_API_KEY = ""
if MODE == "Live Gemini":
    GEMINI_API_KEY = getpass.getpass('Paste your Gemini API key (hidden). Leave blank to use the OC Brain: ').strip()

print(f'Mode: {MODE}   Model: {MODEL}')

In [ ]:
#@title 3 · Load the invoices, the policy, and the OC Brain  { display-mode: "form" }
# Pull the invoice pack and the approval policy from the hosted folder, split the invoice
# PDF into one block of text per invoice, and read the PO register out of the policy.

def fetch_pdf_pages(url):
    r = requests.get(url, timeout=30); r.raise_for_status()
    reader = PdfReader(io.BytesIO(r.content))
    return [(p.extract_text() or '') for p in reader.pages]

invoice_pages = fetch_pdf_pages(INVOICES_URL)
INVOICE_BLOCKS = []
for t in invoice_pages:
    if re.search(r'Invoice \d+ of \d+', t):
        INVOICE_BLOCKS.append(re.sub(r'Invoice \d+ of \d+\s*', '', t).strip())

POLICY_TEXT = '\n'.join(fetch_pdf_pages(POLICY_URL))

# The OC Brain is fetched too - a prepared results file the notebook CALLS if the live model stalls.
OC_BRAIN = requests.get(OC_BRAIN_URL, timeout=30).json()

# Read the purchase-order register out of the policy (one PO per line). Fall back to the
# OC Brain's copy if the text extraction is thin.
PO_INDEX = {}
for m in re.findall(r'(PO-\d+)\s*\|\s*(.+?)\s*\|\s*(.+?)\s*\|\s*Rs\s*([\d,]+)\s*\|\s*(OPEN|CLOSED)', POLICY_TEXT):
    PO_INDEX[m[0]] = {'po': m[0], 'vendor': m[1].strip(), 'item': m[2].strip(),
                      'amount': int(m[3].replace(',', '')), 'status': m[4]}
if len(PO_INDEX) < 3:
    for p in OC_BRAIN['po_register']:
        PO_INDEX[p['po']] = p

print(f'Loaded {len(INVOICE_BLOCKS)} invoices and {len(PO_INDEX)} purchase orders from the hosted pack.')
print(f"OC Brain on standby - {len(OC_BRAIN['invoices'])} prepared results loaded.")

In [ ]:
#@title 4 · The engine (and the OC Brain resilience layer)  { display-mode: "form" }
# Decide the run mode, wire up Gemini if needed, and define one careful call that
# retries on the ordinary errors (429 busy/quota, 503 overloaded).

USE_OC_BRAIN = (MODE == 'Switch to OC Brain') or (not GEMINI_API_KEY)
if USE_OC_BRAIN and MODE == 'Live Gemini':
    print('No API key given - running on the OC Brain.')

model = None
if not USE_OC_BRAIN:
    import google.generativeai as genai
    genai.configure(api_key=GEMINI_API_KEY)
    model = genai.GenerativeModel(MODEL)

def call_gemini(prompt, tries=4):
    """Call the model, backing off on the errors you'll actually meet."""
    last = None
    for i in range(tries):
        try:
            return model.generate_content(prompt).text
        except Exception as e:
            last = e; msg = str(e)
            if '429' in msg or 'RESOURCE_EXHAUSTED' in msg: what = '429 (busy / free-tier limit)'
            elif '503' in msg or 'UNAVAILABLE' in msg:      what = '503 (model overloaded)'
            elif '404' in msg:                              what = '404 (model name)'
            else:                                           what = 'an error'
            wait = 2 ** i
            print(f'Gemini returned {what} - try {i+1}/{tries}. Waiting {wait}s...')
            time.sleep(wait)
    raise last

def extract_json(text):
    """Pull a JSON array/object out of the model's reply."""
    text = text.strip()
    text = re.sub(r'^```(?:json)?', '', text).strip()
    text = re.sub(r'```$', '', text).strip()
    start = min([i for i in (text.find('['), text.find('{')) if i != -1])
    return json.loads(text[start:])

def switch_to_oc_brain(reason):
    global USE_OC_BRAIN
    print(f'{reason} Switching to the OC Brain to keep going.')
    USE_OC_BRAIN = True

print('Engine ready.', '(OC Brain)' if USE_OC_BRAIN else f'(Live: {MODEL})')

## Agent 1 - Ingest & Extract
Turn 25 messy invoices into one clean table of facts. Live, Gemini reads each invoice; on the OC Brain, the same facts come from the prepared file.

In [ ]:
#@title 5 · Agent 1 runs  { display-mode: "form" }
records = []  # one row per invoice, kept in pack order (invoice numbers can repeat!)

if not USE_OC_BRAIN:
    try:
        numbered = '\n\n'.join(f'--- INVOICE {i+1} ---\n{t}' for i, t in enumerate(INVOICE_BLOCKS))
        prompt = (
            'You are Agent 1 in an accounts-payable pipeline. For EACH invoice below, extract exactly '
            'these fields and return ONLY a JSON array, in the same order (no prose):\n'
            '  invoice_number (string), vendor (string), date (string or null if absent), '
            'gstin (string or null if absent), po_referenced (string like "PO-1001", or null if none), '
            'currency ("INR" or "USD"), subtotal (number), gst (number), total (number).\n\n' + numbered
        )
        for d in extract_json(call_gemini(prompt)):
            records.append({'number': d.get('invoice_number'), 'vendor': d.get('vendor'),
                            'extract': {k: d.get(k) for k in ('date','gstin','po_referenced','currency','subtotal','gst','total')}})
    except Exception as e:
        switch_to_oc_brain(f'The live model did not come through ({str(e)[:60]}).')

if USE_OC_BRAIN:
    records = []
    for inv in OC_BRAIN['invoices']:
        ex = inv['extract']
        records.append({'number': ex['invoice_number'], 'vendor': ex['vendor'],
                        'extract': {k: ex.get(k) for k in ('date','gstin','po_referenced','currency','subtotal','gst','total')}})

df1 = pd.DataFrame([{'Invoice': r['number'], 'Vendor': r['vendor'], 'PO': r['extract']['po_referenced'],
                     'Subtotal': r['extract']['subtotal'], 'GST': r['extract']['gst'], 'Total': r['extract']['total']} for r in records])
print(f'Agent 1 extracted {len(df1)} invoices:')
df1

## Agent 2 - Judge & Validate  (rules in code)
Check every invoice against the policy and the PO register. This is money, so the judgement is **deterministic code**, not the model: PO match, vendor match, a 2% tolerance, duplicates, required fields, GST, and the Rs 1,00,000 auto-approve limit.

In [ ]:
#@title 6 · Agent 2 runs  { display-mode: "form" }
# OUR rules, enforced here in code.
AUTO_THRESHOLD = 100000; TOL_PCT = 2; GST_RATE = 0.18
SUFFIX = {'pvt','ltd','llp','inc','private','limited','co','corp','plc','the'}
def norm_vendor(v):
    if not v: return ''
    return ' '.join(t for t in re.sub(r'[^a-z0-9 ]', ' ', str(v).lower()).split() if t not in SUFFIX)
def vendor_matches(a, b):
    na, nb = norm_vendor(a), norm_vendor(b)
    return bool(na) and bool(nb) and (na == nb or na in nb or nb in na)

def validate(rec, po_index, seen_numbers, seen_pairs):
    ex = rec['extract']; reasons = []; matched = None
    num = rec['number']; vendor = rec['vendor']
    sub = ex.get('subtotal'); tot = ex.get('total'); gst = ex.get('gst')
    numeric = isinstance(sub, (int, float))
    foreign = str(ex.get('currency') or 'INR').upper() not in ('INR', 'RS', 'RS.')
    if foreign:
        reasons.append('Invoice is in a foreign currency; policy requires INR.')
    missing = [lbl for lbl, val in [('invoice number', num), ('date', ex.get('date')),
               ('vendor', vendor), ('GSTIN', ex.get('gstin')), ('total', tot)] if not val]
    if missing:
        reasons.append('Missing required field(s): ' + ', '.join(missing) + '.')
    if numeric and sub < 0:
        reasons.append('Credit note / negative amount; a human must handle this.')
    po = ex.get('po_referenced')
    if not po:
        reasons.append('No purchase order referenced.')
    elif po not in po_index:
        reasons.append('PO ' + str(po) + ' is not in the register.')
    else:
        matched = po_index[po]
        if matched['status'] != 'OPEN':
            reasons.append('PO ' + po + ' is ' + matched['status'].lower() + ' (already fully invoiced).')
        if not vendor_matches(vendor, matched['vendor']):
            reasons.append('Vendor does not match PO ' + po + ' (' + matched['vendor'] + ').')
        if (not foreign) and numeric and sub >= 0 and matched.get('amount'):
            diff = abs(sub - matched['amount']) / matched['amount'] * 100
            if diff > TOL_PCT:
                reasons.append('Amount is ' + str(round(diff)) + '% off PO ' + po + ' (Rs ' + format(matched['amount'], ',') + '); limit is ' + str(TOL_PCT) + '%.')
    if (not foreign) and numeric and sub >= 0 and ex.get('gstin'):
        exp = round(GST_RATE * sub)
        if abs((gst or 0) - exp) > 1:
            reasons.append('GST is not 18% of the subtotal (expected Rs ' + format(exp, ',') + ').')
    if num in seen_numbers:
        reasons.append('Duplicate invoice number (' + str(num) + ' already submitted).')
    elif (norm_vendor(vendor), sub) in seen_pairs:
        reasons.append('Looks like a duplicate of an earlier invoice (same vendor and amount).')
    if (not foreign) and isinstance(tot, (int, float)) and tot > AUTO_THRESHOLD:
        reasons.append('Total exceeds the Rs ' + format(AUTO_THRESHOLD, ',') + ' auto-approve limit; needs senior sign-off.')
    return ('Auto-approve' if not reasons else 'Review'), reasons, matched

seen_numbers = set(); seen_pairs = set()
APPROVE = []; REVIEW = []
for r in records:
    decision, reasons, matched = validate(r, PO_INDEX, seen_numbers, seen_pairs)
    r['decision'] = decision; r['reasons'] = reasons
    (APPROVE if decision == 'Auto-approve' else REVIEW).append(r)
    seen_numbers.add(r['number']); seen_pairs.add((norm_vendor(r['vendor']), r['extract'].get('subtotal')))

show = pd.DataFrame([{'Invoice': r['number'], 'Vendor': r['vendor'], 'Decision': r['decision'],
                      'Reason': (r['reasons'][0] if r['reasons'] else '-')} for r in records])
print(f'Agent 2 cleared {len(APPROVE)} to pay and sent {len(REVIEW)} to a human:')
show

In [ ]:
#@title    ...the split, in plain terms  { display-mode: "form" }
print(f'CLEAR TO PAY ({len(APPROVE)}):')
for r in APPROVE:
    print(f"  {r['number']}  {r['vendor']}  Rs {int(r['extract']['total']):,}")
print(f'\nFOR A HUMAN TO REVIEW ({len(REVIEW)}):')
for r in REVIEW:
    print(f"  {r['number']}  {r['vendor']}  ->  {r['reasons'][0]}")

## Agent 3 - Communicate  (behind a human gate)
Draft a summary to the finance manager. The agent **recommends**; it never pays. You decide.

In [ ]:
#@title 7 · Agent 3 drafts (nothing is paid)  { display-mode: "form" }
draft = None
if not USE_OC_BRAIN:
    try:
        approve = [{'invoice': r['number'], 'vendor': r['vendor'], 'total': r['extract']['total']} for r in APPROVE]
        review = [{'invoice': r['number'], 'vendor': r['vendor'], 'reason': r['reasons'][0]} for r in REVIEW]
        prompt = (
            'You are Agent 3 in an accounts-payable pipeline. Write a short, professional email to the '
            'Finance Manager summarising an invoice run. State how many are clear to pay and list them '
            'briefly, then list the ones held for review with their one-line reason. Make clear that '
            'nothing has been paid and that final approval is theirs. Return only the email text.\n\n'
            + json.dumps({'clear_to_pay': approve, 'for_review': review})
        )
        draft = call_gemini(prompt)
    except Exception as e:
        switch_to_oc_brain(f'The live model did not come through ({str(e)[:60]}).')
if USE_OC_BRAIN or draft is None:
    draft = OC_BRAIN['draft']

print('=' * 70)
print(draft)
print('=' * 70)
print('\nHUMAN GATE: this is a draft. The agent has paid nothing and sent nothing. You decide what happens next.')

In [ ]:
#@title 8 · Your decision  { display-mode: "form" }
DECISION = "Hold"  #@param ["Hold", "Approve the clear-to-pay list (I will release it myself)", "Reject"]
if DECISION.startswith('Approve'):
    print('You approved the clear-to-pay list. The agent still pays nothing - you release it yourself, deliberately.')
elif DECISION == 'Reject':
    print('Rejected. Nothing leaves this notebook.')
else:
    print('On hold. Nothing is paid. The decision stays with you.')

## Keep the result
Save the run and the draft to files you can download or paste into your notes, and compare the agent's split with the one the room worked out by hand.

In [ ]:
#@title 9 · Save the invoice run  { display-mode: "form" }
lines = ['# Invoice run - FinOps Invoicing', '',
         f'Mode: {"OC Brain" if USE_OC_BRAIN else MODEL}', '',
         f'## Clear to pay ({len(APPROVE)})', '']
for r in APPROVE:
    lines.append(f"- {r['number']} - {r['vendor']} - Rs {int(r['extract']['total']):,}")
lines += ['', f'## For a human to review ({len(REVIEW)})', '']
for r in REVIEW:
    lines.append(f"- {r['number']} - {r['vendor']} - {r['reasons'][0]}")
lines += ['', '## Draft to the finance manager', '', '```', draft, '```']
md_out = '\n'.join(lines)

with open('invoicing_summary.md', 'w') as f: f.write(md_out)
with open('invoicing_results.json', 'w') as f:
    json.dump({'mode': 'oc_brain' if USE_OC_BRAIN else MODEL,
               'clear_to_pay': [{'invoice': r['number'], 'vendor': r['vendor'], 'total': r['extract']['total']} for r in APPROVE],
               'for_review': [{'invoice': r['number'], 'vendor': r['vendor'], 'reason': r['reasons'][0]} for r in REVIEW],
               'draft': draft}, f, indent=2)

print('Saved: invoicing_summary.md and invoicing_results.json (open them from the file panel on the left).')
print('\n----- copy from here -----\n')
print(md_out)
# To download instead, uncomment:
# from google.colab import files; files.download('invoicing_summary.md')

## What you just saw
The same three narrow agents, chained: extract, judge, communicate - only now the judge is pure **code**, because money leaves no room for vibes, and the final call still belongs to a **human**. Swap this pack for your own and the pipeline serves a different job, which is exactly what you'll do next with your own use-case.